#Random Forest Regressor

In [4]:
import numpy as np
import pandas as pd
import joblib

from scipy import stats
from scipy.stats import randint

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (
    train_test_split,
    StratifiedShuffleSplit,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.metrics import mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor

In [5]:
housing = fetch_california_housing(as_frame=True)
housing_df = housing.frame.copy()

# Create a target column name that matches the textbook section
housing_df["median_house_value"] = housing_df["MedHouseVal"]

# Create a simple categorical feature so one-hot encoding and feature importance make sense
housing_df["ocean_proximity"] = pd.cut(
    housing_df["Latitude"],
    bins=[-np.inf, 33, 35, 37, np.inf],
    labels=["NEAR BAY", "INLAND", "NEAR OCEAN", "ISLAND"]
).astype(str)

housing_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,median_house_value,ocean_proximity
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,4.526,ISLAND
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,3.585,ISLAND
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,3.521,ISLAND
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,3.413,ISLAND
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,3.422,ISLAND


In [6]:
housing_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   MedInc              20640 non-null  float64
 1   HouseAge            20640 non-null  float64
 2   AveRooms            20640 non-null  float64
 3   AveBedrms           20640 non-null  float64
 4   Population          20640 non-null  float64
 5   AveOccup            20640 non-null  float64
 6   Latitude            20640 non-null  float64
 7   Longitude           20640 non-null  float64
 8   MedHouseVal         20640 non-null  float64
 9   median_house_value  20640 non-null  float64
 10  ocean_proximity     20640 non-null  object 
dtypes: float64(10), object(1)
memory usage: 1.7+ MB


In [7]:
# Separate predictors and labels
housing = housing_df.drop("median_house_value", axis=1)
housing_labels = housing_df["median_house_value"].copy()

housing.shape, housing_labels.shape

((20640, 10), (20640,))

In [8]:
# Create an income category for stratified splitting
housing["income_cat"] = pd.cut(
    housing["MedInc"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

housing["income_cat"].value_counts().sort_index()

,count
income_cat,
1,822
2,6581
3,7236
4,3639
5,2362


In [9]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.iloc[train_index].copy()
    strat_test_set = housing.iloc[test_index].copy()

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

strat_train_set.shape, strat_test_set.shape

((16512, 10), (4128, 10))

In [10]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.iloc[train_index].copy()
    strat_test_set = housing.iloc[test_index].copy()

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

strat_train_set.shape, strat_test_set.shape

((16512, 10), (4128, 10))

In [11]:
housing = strat_train_set.drop("median_house_value", axis=1, errors="ignore").copy()
if "median_house_value" in strat_train_set.columns:
    housing_labels = strat_train_set["median_house_value"].copy()
else:
    housing_labels = housing_df.loc[housing.index, "median_house_value"].copy()

housing.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,ocean_proximity
12655,2.1736,29.0,5.485836,1.128895,2237.0,3.168555,38.52,-121.46,0.721,ISLAND
15502,6.3373,7.0,6.927083,1.113281,2015.0,2.623698,33.09,-117.23,2.796,INLAND
2908,2.8750,44.0,5.393333,1.033333,667.0,2.223333,35.37,-119.04,0.827,NEAR OCEAN
14053,2.2264,24.0,3.886128,1.074534,898.0,1.859213,32.75,-117.13,1.125,NEAR BAY
20496,4.4964,27.0,6.096552,1.113793,1837.0,3.167241,34.28,-118.70,2.383,INLAND


In [12]:
def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=42):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None):
        self.kmeans_ = KMeans(n_clusters=self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X)
        return self

    def transform(self, X):
        X = np.asarray(X)
        centers = self.kmeans_.cluster_centers_
        sq_dist = ((X[:, np.newaxis, :] - centers[np.newaxis, :, :]) ** 2).sum(axis=2)
        return np.exp(-self.gamma * sq_dist)

    def get_feature_names_out(self, names=None):
        return [f"geo__Cluster {i} similarity" for i in range(self.n_clusters)]

In [13]:
# Column groups
geo_attribs = ["Latitude", "Longitude"]
cat_attribs = ["ocean_proximity"]

num_attribs = [col for col in housing.columns if col not in geo_attribs + cat_attribs]
num_attribs

['MedInc',
 'HouseAge',
 'AveRooms',
 'AveBedrms',
 'Population',
 'AveOccup',
 'MedHouseVal']

In [14]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessing = ColumnTransformer([
    ("geo", ClusterSimilarity(n_clusters=10, gamma=1.0, random_state=42), geo_attribs),
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

preprocessing

ColumnTransformer(transformers=[('geo', ClusterSimilarity(),
                                 ['Latitude', 'Longitude']),
                                ('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
                                  'Population', 'AveOccup', 'MedHouseVal']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['ocean_proximity'])])

In [18]:
housing_prepared = preprocessing.fit_transform(housing)
housing_prepared.shape

(16512, 21)

In [19]:
forest_reg = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42)),
])

forest_rmses = -cross_val_score(
    forest_reg,
    housing,
    housing_labels,
    scoring="neg_root_mean_squared_error",
    cv=10
)

pd.Series(forest_rmses).describe()

,0
count,10.000000
mean,0.000783
std,0.000486
min,0.000445
25%,0.000523
50%,0.000562
75%,0.000915
max,0.002043


In [20]:
full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42)),
])

param_grid = [
    {
        "preprocessing__geo__n_clusters": [5, 8, 10],
        "random_forest__max_features": [4, 6, 8],
    },
    {
        "preprocessing__geo__n_clusters": [10, 15],
        "random_forest__max_features": [6, 8, 10],
    },
]

grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error"
)

grid_search.fit(housing, housing_labels)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('geo',
                                                                         ClusterSimilarity(),
                                                                         ['Latitude',
                                                                          'Longitude']),
                                                                        ('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['MedInc',
                                                                          'HouseAge',
                                                                          'AveRooms',
                                                                          'AveBedrms',
                                                                          'Population',
                                                                          'AveOccup',
                                                                          'MedHouseVal']),
                                                                        ('cat',
                                                                         Pipeline(steps=...
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['ocean_proximity'])])),
                                       ('random_forest',
                                        RandomForestRegressor(random_state=42))]),
             param_grid=[{'preprocessing__geo__n_clusters': [5, 8, 10],
                          'random_forest__max_features': [4, 6, 8]},
                         {'preprocessing__geo__n_clusters': [10, 15],
                          'random_forest__max_features': [6, 8, 10]}],
             scoring='neg_root_mean_squared_error')

In [21]:
grid_search.best_params_

{'preprocessing__geo__n_clusters': 5, 'random_forest__max_features': 8}

In [22]:
cv_res = pd.DataFrame(grid_search.cv_results_).copy()

cv_res["mean_test_rmse"] = -cv_res["mean_test_score"]
cv_res["split0"] = -cv_res["split0_test_score"]
cv_res["split1"] = -cv_res["split1_test_score"]
cv_res["split2"] = -cv_res["split2_test_score"]

cv_res = cv_res[[
    "param_preprocessing__geo__n_clusters",
    "param_random_forest__max_features",
    "split0",
    "split1",
    "split2",
    "mean_test_rmse"
]].rename(columns={
    "param_preprocessing__geo__n_clusters": "n_clusters",
    "param_random_forest__max_features": "max_features",
})

cv_res.sort_values(by="mean_test_rmse", ascending=True).head()

,n_clusters,max_features,split0,split1,split2,mean_test_rmse
2,5,8,0.018695,0.018853,0.017851,0.018467
11,10,10,0.018944,0.018573,0.019150,0.018889
5,8,8,0.026926,0.029931,0.029871,0.028909
8,10,8,0.035324,0.039208,0.037546,0.037359
10,10,8,0.035324,0.039208,0.037546,0.037359


In [23]:
param_distribs = {
    "preprocessing__geo__n_clusters": randint(low=3, high=50),
    "random_forest__max_features": randint(low=2, high=20),
}

rnd_search = RandomizedSearchCV(
    full_pipeline,
    param_distributions=param_distribs,
    n_iter=10,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42
)

rnd_search.fit(housing, housing_labels)

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('geo',
                                                                               ClusterSimilarity(),
                                                                               ['Latitude',
                                                                                'Longitude']),
                                                                              ('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               ['MedInc',
                                                                                'HouseAge',
                                                                                'AveRooms',
                                                                                'AveBedrms',
                                                                                'Population',
                                                                                'AveOccup',
                                                                                'MedHouseVal']),
                                                                              ('cat',
                                                                               Pipeline(...
                                             ('random_forest',
                                              RandomForestRegressor(random_state=42))]),
                   param_distributions={'preprocessing__geo__n_clusters': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7d151ca0faa0>,
                                        'random_forest__max_features': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7d1521a5d400>},
                   random_state=42, scoring='neg_root_mean_squared_error')

In [24]:
rnd_search.best_params_

{'preprocessing__geo__n_clusters': 21, 'random_forest__max_features': 12}

In [25]:
final_model = rnd_search.best_estimator_

feature_importances = final_model["random_forest"].feature_importances_
feature_importances.round(2)

array([0.  , 0.01, 0.01, 0.  , 0.  , 0.  , 0.02, 0.  , 0.  , 0.  , 0.01,
       0.01, 0.  , 0.  , 0.  , 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.16,
       0.  , 0.03, 0.  , 0.  , 0.02, 0.69, 0.  , 0.  , 0.  , 0.  ])

In [26]:
sorted(
    zip(feature_importances, final_model["preprocessing"].get_feature_names_out()),
    reverse=True
)[:20]

[(np.float64(0.6868197077019501), 'num__MedHouseVal'),
 (np.float64(0.1581654194210408), 'num__MedInc'),
 (np.float64(0.030420682422800847), 'num__AveRooms'),
 (np.float64(0.016510356844067993), 'geo__geo__Cluster 6 similarity'),
 (np.float64(0.015815947513044534), 'num__AveOccup'),
 (np.float64(0.012342803207563946), 'geo__geo__Cluster 15 similarity'),
 (np.float64(0.010942324567336709), 'geo__geo__Cluster 2 similarity'),
 (np.float64(0.009199991090078744), 'geo__geo__Cluster 10 similarity'),
 (np.float64(0.00809226198478124), 'geo__geo__Cluster 11 similarity'),
 (np.float64(0.007474055645217481), 'geo__geo__Cluster 20 similarity'),
 (np.float64(0.005489180465772387), 'geo__geo__Cluster 1 similarity'),
 (np.float64(0.004285988988433854), 'geo__geo__Cluster 13 similarity'),
 (np.float64(0.003793190896002604), 'geo__geo__Cluster 17 similarity'),
 (np.float64(0.003453222855610883), 'num__HouseAge'),
 (np.float64(0.00322966811462806), 'geo__geo__Cluster 0 similarity'),
 (np.float64(0.0031

In [29]:
# Recreate proper test predictors/labels
X_test = strat_test_set.copy()
if "median_house_value" in X_test.columns:
    X_test = X_test.drop("median_house_value", axis=1)
    y_test = strat_test_set["median_house_value"].copy()
else:
    y_test = housing_df.loc[X_test.index, "median_house_value"].copy()

final_predictions = final_model.predict(X_test)

final_rmse = np.sqrt(mean_squared_error(y_test, final_predictions))
final_rmse

np.float64(0.03776491368554853)

In [30]:
confidence = 0.95
squared_errors = (final_predictions - y_test) ** 2

np.sqrt(stats.t.interval(
    confidence,
    len(squared_errors) - 1,
    loc=squared_errors.mean(),
    scale=stats.sem(squared_errors)
))

array([0.03174648, 0.04294809])

In [31]:
joblib.dump(final_model, "my_california_housing_model.pkl")

['my_california_housing_model.pkl']

In [32]:
final_model_reloaded = joblib.load("my_california_housing_model.pkl")
predictions = final_model_reloaded.predict(X_test.iloc[:5])
predictions

array([4.9989698, 1.63003  , 2.04417  , 1.60984  , 1.84376  ])